# NIG-TimeNet (PyTorch): build the training data

**Runtime > Run all.** The only cell with settings is the next one; every option of the data pipeline is a dotted key in
`CHANGES` (see `nig_timenet_torch/data/settings.py`; `python -m nig_timenet_torch.data.cli show` prints them all).
The raw files are the ones `crypto-signal-prediction` uses: `<RAW_ROOT>/history_1h/<SYMBOL>.csv(.gz)` and
`<RAW_ROOT>/crypto_data/asset_registry.csv`. The result is written as `<NAME>_<stamp>.dataset` and `<NAME>_latest.dataset`
in `OUT_DIR`; the training notebooks read it by `NAME` (the `_latest` copy) or by the full dated name.

In [ ]:
TORCH_REPO, TORCH_REF = "yuosef772424/nig-timenet-torch", "feat/native-data-notebooks-uncertainty"
EXPECT_DEVICE = "cpu"                         # "cpu" only for a deliberate CPU run; a cuda run on a CPU VM fails
TOKEN_PATH = "/content/drive/MyDrive/crypto/secrets/github_token.txt"
WORKDIR = "/content/nig_runner"                # clones' scratch, logs, git credentials helper: never /tmp (rule E2)
TORCH_DIR = "/content/nig-timenet-torch"
DRIVE_RUNS = "/content/drive/MyDrive/nig_timenet_runs"
EXTRA_MODULES = (("pyarrow", "pyarrow"), ("pandas_ta_classic", "pandas-ta-classic==0.8.32"), ("requests", "requests"))
RAW_ROOT = "/content/drive/MyDrive"            # holds history_1h/ and crypto_data/asset_registry.csv
OUT_DIR = "/content/drive/MyDrive/crypto_model/preprocessed_data"
NAME = "preprocessing_output"                  # files: <NAME>_<stamp>.dataset and <NAME>_latest.dataset
PRESETS = ("hourly_w32_s8",)                   # data/presets.py: hourly_w32_s8, hourly_4h_closed, dated_splits, tiny
CHANGES = {}                                   # e.g. {"windows.stride": 4, "source.categories": ("all_global",)}
FETCH = ""                                     # "futures" or "vision": download the newest candles first; "" = read only

## 1. Setup
The first cell is plain code: nothing of the package exists on the VM yet, so it downloads the one stdlib-only file
`colab/bootstrap.py` with the token and runs it. Everything after it is a normal import.

In [ ]:
import getpass, os, urllib.error, urllib.request
from google.colab import drive
drive.mount("/content/drive")
token = open(TOKEN_PATH).read().strip() if os.path.exists(TOKEN_PATH) else getpass.getpass("GitHub token (not shown): ")
url = f"https://api.github.com/repos/{TORCH_REPO}/contents/src/nig_timenet_torch/colab/bootstrap.py?ref={TORCH_REF}"
req = urllib.request.Request(url, headers={"Authorization": "Bearer " + token.strip(), "User-Agent": "nig-colab",
                                           "Accept": "application/vnd.github.raw"})
try:
    boot_source = urllib.request.urlopen(req, timeout=60).read().decode()
except urllib.error.HTTPError as e:
    raise SystemExit(f"GitHub answered HTTP {e.code}: the token is missing, expired or cannot read {TORCH_REPO}. "
                     f"Delete {TOKEN_PATH} and run this cell again.")
boot = {}
exec(compile(boot_source, "bootstrap.py", "exec"), boot)
token = boot["acquire_token"](TOKEN_PATH, TORCH_REPO, lambda: getpass.getpass("GitHub token (not shown): "))

In [ ]:
shas = boot["setup_clones"](token, WORKDIR, {"torch": (TORCH_REPO, TORCH_REF, TORCH_DIR)})
install = boot["install_package"](TORCH_DIR, modules=EXTRA_MODULES)

In [ ]:
from nig_timenet_torch.colab.session import check_environment
setup = check_environment(EXPECT_DEVICE, shas, install, WORKDIR, DRIVE_RUNS)

## 2. Settings
`preset(...)` applies the named patches in order, then `CHANGES` on top. Unknown or misspelt keys fail with a
suggestion, before anything is built.

In [ ]:
from nig_timenet_torch.data.presets import preset
settings = preset(*PRESETS).updated({"source.root": RAW_ROOT, "output.directory": OUT_DIR, "output.name": NAME, "source.fetch": FETCH, **CHANGES})
coins = len(settings.source.coins) or settings.source.categories or "all"
print(settings.windows, coins)

## 3. Download (optional)
Only when `FETCH` is set: appends the newest hourly candles to the raw files, then updates the registry.

In [ ]:
from nig_timenet_torch.data.fetch import fetch_all
from nig_timenet_torch.data.sources import resolve_coins
added = fetch_all(settings.source, resolve_coins(settings.source)) if FETCH else {}
print(sum(added.values()), "rows added for", len(added), "coins")

## 4. Build and write
Builds every window, target and scaler exactly as `crypto-signal-prediction` does (the tests compare them bit for bit),
drops the sealed holdout coins and period, writes the dated folder and `_latest`, and prints a summary.

In [ ]:
from nig_timenet_torch.data.pipeline import build_artifact, summary_lines
dataset, manifest = build_artifact(settings)
print("\n".join(summary_lines(dataset)))
print(manifest["export"]["rows"], "rows, max bar time", manifest["export"]["max_bar_time"])

## 5. Check
Reads the file back the way the training notebooks do and re-hashes every array.

In [ ]:
from nig_timenet_torch.data_io.loading import LoaderSettings, load_dataset
loaded = load_dataset(LoaderSettings(directory=OUT_DIR, name=NAME, verify_hashes=True))
print(loaded.path, loaded.format, len(loaded.dataset["last_candles"]), "windows")